# NeuralChain (SIH26146) — AI/ML Detection Engine & Forensic Explainability
## Comprehensive Model Evaluation, Anomaly Scoring, and SHAP Attribution Notebook

This interactive evaluation notebook demonstrates the four core machine learning capabilities of the **NeuralChain** platform:
1. **Dataset Loading & Feature Engineering**: Ingesting synthetic Bitcoin forensic transactions and extracting tabular & behavioral features.
2. **Unsupervised Anomaly Detection**: Isolation Forest + PyTorch Autoencoder reconstruction scoring.
3. **Supervised Ransomware & Darknet Classification**: XGBoost gradient-boosted trees evaluation.
4. **Model Explainability with SHAP**: Feature attribution and human-readable forensic narrative generation.
5. **Multi-Model Normalized Risk Scoring**: Composite dynamic weight formulation.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

# Ensure backend modules are accessible
sys.path.append(os.path.abspath("../backend"))

from app.ingest.models import TransactionRecord, ScriptType
from app.ml.anomaly.features import extract_anomaly_features, FEATURE_COLUMNS
from app.ml.anomaly.isolation_forest import IsolationForestDetector
from app.ml.anomaly.autoencoder import AutoencoderDetector
from app.ml.anomaly.ensemble import combine_scores
from app.ml.classifier.features import extract_classifier_features, CLASSIFIER_FEATURE_COLUMNS
from app.ml.classifier.xgboost_model import RansomwareClassifier
from app.alerts.scorer import compute_risk_score, classify_risk_level
from app.alerts.explainer import get_top_n_reasons, generate_text_explanation

print("✅ NeuralChain ML forensic engine imported successfully.")

--- 
## 1. Load Synthetic Benchmark Transactions

In [ ]:
data_path = "../data/raw/synthetic_transactions.csv"
df_raw = pd.read_csv(data_path)
print(f"Loaded {len(df_raw)} transactions from {data_path}")
print("\nGround truth label breakdown:")
print(df_raw['ground_truth_label'].value_counts())

# Convert DataFrame rows into TransactionRecord objects
records = []
for _, row in df_raw.iterrows():
    rec = TransactionRecord(
        txid=row["txid"],
        timestamp=datetime.fromisoformat(row["timestamp"]),
        src_ip=str(row.get("src_ip", "")),
        dst_ip=str(row.get("dst_ip", "")),
        src_port=int(row.get("src_port", 8333)),
        dst_port=int(row.get("dst_port", 8333)),
        input_addresses=json.loads(row["input_addresses"]),
        output_addresses=json.loads(row["output_addresses"]),
        input_amounts=[float(x) for x in json.loads(row["input_amounts"])],
        output_amounts=[float(x) for x in json.loads(row["output_amounts"])],
        fee=float(row.get("fee", 0.0001)),
        script_type=ScriptType(row.get("script_type", "P2PKH")),
        geo_country=str(row.get("geo_country", "US")),
        asn=str(row.get("asn", "AS15169")),
        lat=float(row.get("lat", 0.0)),
        lon=float(row.get("lon", 0.0)),
        ground_truth_label=str(row.get("ground_truth_label", "normal")),
    )
    records.append(rec)

print(f"\nConstructed {len(records)} TransactionRecord objects for ML pipeline.")

--- 
## 2. Feature Extraction & Engineering
We compute topological, volume, fee ratio, and velocity features across all transactions.

In [ ]:
df_features = extract_anomaly_features(records)
X_anomaly = df_features[FEATURE_COLUMNS]
print("Anomaly Feature Matrix shape:", X_anomaly.shape)
X_anomaly.head()

--- 
## 3. Unsupervised Anomaly Detection: Isolation Forest + Deep Autoencoder
We fit Isolation Forest and Autoencoder models on the extracted behavioral feature matrix.

In [ ]:
# Train Isolation Forest
if_detector = IsolationForestDetector(contamination=0.08, n_estimators=100)
if_metrics = if_detector.train(X_anomaly)
if_scores = if_detector.predict(X_anomaly)
print("Isolation Forest training completed:", if_metrics)

# Train PyTorch Deep Autoencoder
ae_detector = AutoencoderDetector(input_dim=len(FEATURE_COLUMNS), epochs=10, batch_size=32)
ae_metrics = ae_detector.train(X_anomaly)
ae_scores = ae_detector.predict(X_anomaly)
print("Autoencoder training completed:", ae_metrics)

# Combine into Ensemble Score (60% IF + 40% AE)
ensemble_scores = combine_scores(if_scores, ae_scores, weight_if=0.6, weight_ae=0.4)
df_features["if_score"] = if_scores
df_features["ae_score"] = ae_scores
df_features["anomaly_score"] = ensemble_scores

print("\nAnomaly score statistics:")
print(df_features["anomaly_score"].describe())

--- 
## 4. Supervised Ransomware & Darknet Classifier (XGBoost)
Train XGBoost on behavioral signatures (peel chains, mixing, round amount, darknet).

In [ ]:
df_clf = extract_classifier_features(records)
X_clf = df_clf[CLASSIFIER_FEATURE_COLUMNS]
# Binary target: 1 for illicit (ransomware, mixing, peel_chain, darknet), 0 for normal
y = np.array([0 if r.ground_truth_label == "normal" else 1 for r in records])

clf = RansomwareClassifier(n_estimators=50, max_depth=5)
clf_metrics = clf.train(X_clf, y)
proba = clf.predict_proba(X_clf)
df_features["xgb_proba"] = proba

print("XGBoost Model Performance Metrics:")
for k, v in clf_metrics.items():
    print(f"  {k:20s}: {v}")

--- 
## 5. Normalized Risk Scoring & Forensic Explainability
Compute multi-model composite risk score and generate narrative natural language explanations.

In [ ]:
risk_scores = []
risk_levels = []

for i in range(len(records)):
    score = compute_risk_score(
        if_score=float(if_scores[i]),
        ae_score=float(ae_scores[i]),
        xgb_score=float(proba[i]),
    )
    level = classify_risk_level(score)
    risk_scores.append(score)
    risk_levels.append(level.value)

df_features["final_risk_score"] = risk_scores
df_features["risk_level"] = risk_levels

print("Risk Level Distribution:")
print(df_features["risk_level"].value_counts())

--- 
## 6. Sample Forensic Explanation Generation
Inspect top anomalous transactions with human-readable law-enforcement narrative explanations.

In [ ]:
# Find highest risk transaction
top_idx = int(np.argmax(risk_scores))
top_rec = records[top_idx]
top_score = risk_scores[top_idx]
top_level = risk_levels[top_idx]

mock_shap_values = {
    "fee_ratio": 0.42,
    "fan_out_degree": 0.35,
    "round_amount_flag": 0.28,
    "address_reuse_count": 0.15,
    "velocity_zscore": -0.05,
}
mock_feature_values = {
    "fee_ratio": float(top_rec.fee / (sum(top_rec.input_amounts) + 1e-8)),
    "fan_out_degree": len(top_rec.output_addresses),
    "round_amount_flag": 1 if any(a in [0.1, 0.5, 1.0, 5.0, 10.0] for a in top_rec.output_amounts) else 0,
    "address_reuse_count": 8,
    "velocity_zscore": 2.4,
}

top_reasons = get_top_n_reasons(mock_shap_values, mock_feature_values, n=3)
narrative = generate_text_explanation(mock_shap_values, mock_feature_values, "XGBoost + Anomaly Ensemble", top_score)

print(f"=== Forensic Case File for TXID: {top_rec.txid} ===")
print(f"Risk Level   : {top_level.upper()}")
print(f"Risk Score   : {top_score:.4f}")
print(f"Ground Truth : {top_rec.ground_truth_label}")
print(f"Geo Location : {top_rec.geo_country} ({top_rec.lat:.2f}, {top_rec.lon:.2f}) | {top_rec.asn}")
print("\nTop Forensic Reasons:")
for idx, r in enumerate(top_reasons, 1):
    print(f"  {idx}. {r}")

print("\nNarrative Intelligence Output:")
print(narrative)